# Video to TRC Quickstart

Use this notebook when you have one video file and want pose CSV files plus an OpenSim-friendly TRC export. Run one section at a time and inspect the intermediate tables before continuing.

## 1. Import and set paths

Edit `VIDEO_PATH` and `OUTPUT_DIR` for your project. Keep raw inputs under `data/` and generated files under `outputs/` when possible.

In [ ]:
from pathlib import Path
import monomech as mm

VIDEO_PATH = Path("data/subject01.mp4")
OUTPUT_DIR = Path("outputs/subject01")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("monomech import OK")
print("Available bundled models:", mm.list_builtin_osim_models())

## 2. Load the video trial

The trial object stores the source path and keeps the latest pose results as you estimate them.

In [ ]:
trial = mm.load_video(VIDEO_PATH)
trial

## 3. Estimate pose in stages

`estimate_pose2d()` reads the video and stores both 2D landmarks and MediaPipe world landmarks. The following stages smooth and convert those results into a global marker representation.

In [ ]:
pose2d = trial.estimate_pose2d()
pose3d_world = trial.estimate_pose3d_world()
pose3d_global = trial.estimate_pose3d_global()

print("2D frames:", pose2d.frames)
print("3D landmarks:", len(pose3d_global.landmarks))

## 4. Inspect the result

Look at missing values and the wide table before exporting. If a landmark has too many missing frames, fix the input video or tune the pose stage before moving downstream.

In [ ]:
display(pose2d.summary().head(12))
display(pose3d_global.to_wide_df().head())

## 5. Export CSV and TRC

CSV is useful for inspection and custom analysis. TRC is the handoff format for OpenSim workflows.

In [ ]:
csv_path = pose3d_global.to_csv(OUTPUT_DIR / "subject01_global.csv")
trc_path = pose3d_global.to_trc(OUTPUT_DIR / "subject01_global.trc")

print("CSV:", csv_path)
print("TRC:", trc_path)

## 6. Optional convenience pipeline

Once the staged version looks right, `run_pipeline()` is a shorter wrapper for the same common workflow.

In [ ]:
# run = trial.run_pipeline(export_csv=True, export_trc=True, output_dir=OUTPUT_DIR)
# print(run.csv_paths)
# print(run.trc_path)